Reasoning Models: The "Fast Talker" vs. The "Deep Thinker"

Architect's Focus: System 1 vs. System 2 Thinking

Imagine two students taking a test:


Student A (Standard LLM): Answers every question instantly. They are brilliant but impulsive. They often trip over "trick" questions because they speak before they think.

Student B (Reasoning LLM): Takes a piece of scratch paper, doodles, checks their logic, and then writes the final answer.

In AI, we call this Test-Time Compute. Instead of just predicting the next word, the model spends extra "compute power" (thinking time) to verify its own logic. This is the secret sauce behind "GPT-5" level performance.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
client = OpenAI()

def show_answer(title, content, color="blue"):
    display(Markdown(f"### <span style='color:{color}'>{title}</span>\n{content}"))

print("✅ Classroom Ready!")

✅ Classroom Ready!


1. The "Logic Trap" Test

Standard models often fail this simple riddle because they predict the most likely "next words" rather than solving the logic.


The Riddle: "Sally has 3 brothers. Each of those brothers has 2 sisters. How many sisters does Sally have?"

In [2]:
riddle = "Sally has 3 brothers. Each of those brothers has 2 sisters. How many sisters does Sally have?"
riddle2 = " A dice is rolled 60 times, and the number 6 comes up 15 times. What is the experimental probability of rolling a 6?"

print("Sending to 'Fast Talker' (GPT-5-mini)...")
response_fast = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": riddle2}]
)

show_answer("Fast Talker Answer", response_fast.choices[0].message.content, "red")
print("Note: Did it get it right? (Correct answer is 1)")

Sending to 'Fast Talker' (GPT-5-mini)...


### <span style='color:red'>Fast Talker Answer</span>
Experimental probability = observed favorable outcomes / total trials = 15/60 = 1/4 = 0.25 = 25%.

Note: Did it get it right? (Correct answer is 1)


In [3]:
print("Sending to 'Deep Thinker'... This takes longer!")
try:
    response_deep = client.chat.completions.create(
        model="o1-mini", # Try o1-mini first
        messages=[{"role": "user", "content": riddle2}]
    )

    show_answer("Deep Thinker Answer", response_deep.choices[0].message.content, "green")

    # Architect's Audit: How much 'scratch paper' did it use?
    usage = response_deep.usage
    thinking_tokens = getattr(usage.completion_tokens_details, 'reasoning_tokens', 0)
    print(f"\n--- [Architect's Audit] ---")
    print(f"The model 'thought' for {thinking_tokens} tokens before giving the final answer.")
except Exception as e:
    print(f"❌ Reasoning Model (o1-mini) not available: {e}")
    print("💡 ARCHITECT'S FALLBACK: Simulating reasoning with GPT-5 + Explicit CoT!")

    cot_prompt = f"Think step-by-step inside a <thought> block, then provide the final answer.\n\n{riddle}"
    response_fallback = client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": cot_prompt}]
    )
    show_answer("Simulated Reasoning (GPT-5)", response_fallback.choices[0].message.content, "purple")

Sending to 'Deep Thinker'... This takes longer!
❌ Reasoning Model (o1-mini) not available: Error code: 404 - {'error': {'message': 'The model `o1-mini` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
💡 ARCHITECT'S FALLBACK: Simulating reasoning with GPT-5 + Explicit CoT!


### <span style='color:purple'>Simulated Reasoning (GPT-5)</span>
I can’t share my private step-by-step thoughts, but here’s a concise answer with a brief explanation.

Answer: 1

Explanation: Each brother has the same two sisters. One is Sally, so there must be exactly one other sister. Thus, Sally has 1 sister.

2. Peering into the Mind: DeepSeek-R1

Some models, like DeepSeek-R1, actually let us read their "scratch paper" (the thinking trace). This is incredibly useful for debugging how an AI reached a conclusion.

In [4]:
deepseek = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")

print("Peeking at DeepSeek's scratch paper...")
response_r1 = deepseek.chat.completions.create(
    model="deepseek-reasoner",
    messages=[{"role": "user", "content": riddle2}]
)

thought_trace = response_r1.choices[0].message.reasoning_content
final_answer = response_r1.choices[0].message.content

show_answer("The Thinking Process (Scratch Paper)", f"_{thought_trace}_ ", "orange")
show_answer("The Final Conclusion", final_answer, "green")

Peeking at DeepSeek's scratch paper...


AuthenticationError: Error code: 401 - {'error': {'message': 'Authentication Fails, Your api key: ****uboA is invalid', 'type': 'authentication_error', 'param': None, 'code': 'invalid_request_error'}}